In [13]:
# Task 1:
import sqlalchemy
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import sessionmaker, declarative_base

# 1. Tworzymy silnik - 'sqlite:///students.db' stworzy fizyczny plik w Twoim folderze
engine = create_engine('sqlite:///students.db', echo=True) # echo=True pokaże nam surowy SQL w konsoli!

# 2. We create a database for models
Base = declarative_base()

# 3. Task 2: We define the structure of the Student table
class Student(Base):
    __tablename__ = 'students'
    id = Column(Integer, primary_key=True)
    name = Column(String(100))
    age = Column(Integer)
    email = Column(String(150), unique=True)

# 4. We create a table in a file students.db
Base.metadata.create_all(engine)

# 5. We are opening the "service office", i.e. the session
Session = sessionmaker(bind=engine)
session = Session()

print("✅ students.db database created, students table ready ")

2026-04-25 19:02:33,573 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 19:02:33,574 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("students")
2026-04-25 19:02:33,574 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-25 19:02:33,576 INFO sqlalchemy.engine.Engine COMMIT
✅ students.db database created, students table ready 


In [14]:
# Szybka naprawa sesji i czyszczenie
session.rollback() # Odblokowuje sesję po błędzie UNIQUE [cite: 657]

# Usuwamy wszystkich studentów, żeby móc ich dodać jeszcze raz bez błędu UNIQUE
session.query(Student).delete() 
session.commit()

print("✅ Sesja odblokowana, tabela studenci jest pusta. Możesz teraz uruchomić Zadanie 3!")

# Task 3: Adding 5 students:
student_list = [
    Student(name="Anna", age=20, email="anna@example.com"),
    Student(name="Jan", age=22, email="jan@example.com"),
    Student(name="Kasia", age=19, email="kasia@example.com"),
    Student(name="Marek", age=25, email="marek@example.com"),
    Student(name="Piotr", age=21, email="piotr@example.com")
]

session.add_all(student_list)
session.commit() # We save changes permanently

2026-04-25 19:04:23,855 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 19:04:23,857 INFO sqlalchemy.engine.Engine DELETE FROM students
2026-04-25 19:04:23,857 INFO sqlalchemy.engine.Engine [generated in 0.00053s] ()
2026-04-25 19:04:23,859 INFO sqlalchemy.engine.Engine COMMIT
✅ Sesja odblokowana, tabela studenci jest pusta. Możesz teraz uruchomić Zadanie 3!
2026-04-25 19:04:23,860 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 19:04:23,861 INFO sqlalchemy.engine.Engine INSERT INTO students (name, age, email) VALUES (?, ?, ?) RETURNING id
2026-04-25 19:04:23,862 INFO sqlalchemy.engine.Engine [generated in 0.00007s (insertmanyvalues) 1/5 (ordered; batch not supported)] ('Anna', 20, 'anna@example.com')
2026-04-25 19:04:23,863 INFO sqlalchemy.engine.Engine INSERT INTO students (name, age, email) VALUES (?, ?, ?) RETURNING id
2026-04-25 19:04:23,863 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/5 (ordered; batch not supported)] ('Jan', 22, 'jan@example.com')
2026-0

In [15]:
# Task 4: Data reading
all_students = session.query(Student).all()
print(f"How many students do we have? {len(all_students)}")

# Filtering: find Jan
jan = session.query(Student).filter_by(name="Jan").first()
print(f"student found: {jan.name}, email:{jan.email}")

# older than 20 years old:
adults = session.query(Student).filter(Student.age > 20).all()
print(f"Students 20+: {[s.name for s in adults]}")

2026-04-25 19:04:27,168 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 19:04:27,169 INFO sqlalchemy.engine.Engine SELECT students.id AS students_id, students.name AS students_name, students.age AS students_age, students.email AS students_email 
FROM students
2026-04-25 19:04:27,170 INFO sqlalchemy.engine.Engine [generated in 0.00037s] ()
How many students do we have? 5
2026-04-25 19:04:27,171 INFO sqlalchemy.engine.Engine SELECT students.id AS students_id, students.name AS students_name, students.age AS students_age, students.email AS students_email 
FROM students 
WHERE students.name = ?
 LIMIT ? OFFSET ?
2026-04-25 19:04:27,171 INFO sqlalchemy.engine.Engine [generated in 0.00021s] ('Jan', 1, 0)
student found: Jan, email:jan@example.com
2026-04-25 19:04:27,172 INFO sqlalchemy.engine.Engine SELECT students.id AS students_id, students.name AS students_name, students.age AS students_age, students.email AS students_email 
FROM students 
WHERE students.age > ?
2026-04-25 19:04:2

In [16]:
# Task 5: Update
# Anna's age change
anna = session.query(Student).filter_by(name="Anna").first()
if anna:
    anna.age = 21 #we assign a new value
    session.commit()

# Task 6: Delete
#We are removing Marek (because he is 25 years old, as instructed, we are removing older people)
to_removed = session.query(Student).filter(Student.name == "Marek").first()
if to_removed:
    session.delete(to_removed)
    session.commit()

print("Changes saved. Anna is 21 years old, Marek deleted.")

2026-04-25 19:04:31,144 INFO sqlalchemy.engine.Engine SELECT students.id AS students_id, students.name AS students_name, students.age AS students_age, students.email AS students_email 
FROM students 
WHERE students.name = ?
 LIMIT ? OFFSET ?
2026-04-25 19:04:31,145 INFO sqlalchemy.engine.Engine [cached since 3.974s ago] ('Anna', 1, 0)
2026-04-25 19:04:31,149 INFO sqlalchemy.engine.Engine UPDATE students SET age=? WHERE students.id = ?
2026-04-25 19:04:31,150 INFO sqlalchemy.engine.Engine [generated in 0.00135s] (21, 1)
2026-04-25 19:04:31,151 INFO sqlalchemy.engine.Engine COMMIT
2026-04-25 19:04:31,154 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 19:04:31,155 INFO sqlalchemy.engine.Engine SELECT students.id AS students_id, students.name AS students_name, students.age AS students_age, students.email AS students_email 
FROM students 
WHERE students.name = ?
 LIMIT ? OFFSET ?
2026-04-25 19:04:31,155 INFO sqlalchemy.engine.Engine [cached since 3.984s ago] ('Marek', 1, 0)
2026-

In [17]:
# Task 7
from sqlalchemy import Column, Integer, String, ForeignKey, create_engine
from sqlalchemy.orm import relationship, sessionmaker, declarative_base

# 1. Configuration - Creating a local SQLite database file named 'students.db'
# The engine is the starting point for any SQLAlchemy application.
engine = create_engine('sqlite:///students.db', echo=False)
Base = declarative_base()

# 2. Model Definitions - Establishing a One-to-Many relationship
class Teacher(Base):
    """
    Represents the 'one' side of the relationship. 
    One teacher can be associated with many courses.
    """
    __tablename__ = 'teachers'
    id = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False)
    
    # Relationship: Connects to the Course model. 
    # back_populates ensures that changes are synchronized on both ends.
    courses = relationship("Course", back_populates="teacher")

class Course(Base):
    """
    Represents the 'many' side of the relationship.
    Each course is linked to one specific teacher via a Foreign Key.
    """
    __tablename__ = 'courses'
    id = Column(Integer, primary_key=True)
    title = Column(String(100), nullable=False)
    
    # ForeignKey: Stores the ID of the teacher this course belongs to.
    teacher_id = Column(Integer, ForeignKey('teachers.id'))
    
    # Relationship: Links back to the Teacher model.
    teacher = relationship("Teacher", back_populates="courses")

# 3. Database Creation - Translating Python classes into SQL tables
Base.metadata.create_all(engine)

# 4. Session Setup - Creating a workspace for database operations
Session = sessionmaker(bind=engine)
session = Session()

# 5. Data Entry - Adding a teacher and assigning courses (Zadanie 7)
try:
    # Create an instance of Teacher
    new_teacher = Teacher(name="Dr. Kowalski")
    
    # Create instances of Course and link them to the teacher
    c1 = Course(title="Data Analysis", teacher=new_teacher)
    c2 = Course(title="SQL Basics", teacher=new_teacher)

    # Add objects to the session and save to the database
    session.add_all([new_teacher, c1, c2])
    session.commit()
    
    # Verification - Accessing data through the relationship
    print(f"✅ Success! Teacher {new_teacher.name} manages the following courses:")
    for course in new_teacher.courses:
        print(f" - Course Title: {course.title}")

except Exception as e:
    # If an error occurs, roll back the transaction to maintain data integrity
    session.rollback()
    print(f"❌ An error occurred: {e}")

✅ Success! Teacher Dr. Kowalski manages the following courses:
 - Course Title: Data Analysis
 - Course Title: SQL Basics


In [18]:
from sqlalchemy import func
from sqlalchemy.orm import sessionmaker

# Create a session factory and an actual session instance
Session = sessionmaker(bind=engine) 
session = Session()

# Task 11: Statistical Analysis and Advanced Queries
print("--- Task 11: Reports and Statistics ---")

# 1. Counting total records
# We use func.count() to see how many students we have in total.
total_students = session.query(func.count(Student.id)).scalar()
print(f"Total number of students: {total_students}") # [cite: 1037]

# 2. Calculating Averages
# We calculate the average age of all students. 
# .scalar() returns the single numeric result of the query.
avg_age = session.query(func.avg(Student.age)).scalar()
print(f"Average student age: {avg_age:.2f}") # [cite: 1099]

# 3. Complex Filtering (AND/OR)
# Let's find students who are between 20 and 23 years old.
filtered_students = session.query(Student).filter(
    Student.age >= 20, 
    Student.age <= 23
).all()
print(f"Students aged 20-23: {[s.name for s in filtered_students]}") # [cite: 1036]

# 4. Using LIKE for text searching
# Searching for any student with an email address containing 'example'.
email_search = session.query(Student).filter(Student.email.like('%example%')).count()
print(f"Number of students with 'example' in email: {email_search}") # [cite: 574, 575]

# 5. Grouping and Summing (E-commerce style logic)
# Even though we don't have orders, we can count courses per teacher.
# This mimics 'Sum of orders per customer' logic from the task description.
teacher_stats = session.query(
    Teacher.name, 
    func.count(Course.id).label('course_count')
).join(Course).group_by(Teacher.name).all()

for name, count in teacher_stats:
    print(f"Teacher {name} manages {count} course(s).") # [cite: 1105]

--- Task 11: Reports and Statistics ---
Total number of students: 4
Average student age: 20.75
Students aged 20-23: ['Anna', 'Jan', 'Piotr']
Number of students with 'example' in email: 4
Teacher Dr. Kowalski manages 6 course(s).
